# APOE-ind Set 57 → ARHGAP35 Expression → Tangles (sqrt) Mediation Analysis

## Overview
This notebook performs mediation analysis with two variants in a APOE-independent eQTL set of ARHGAP35 in Ast.10 as the exposure:
1. **chr19_44827033_TGATAGATAGATAGATA_TGATA** → ARHGAP35_expression → tangles_sqrt
2. **chr19_44840322_G_A** → ARHGAP35_expression → tangles_sqrt

## Mediation Model
- **Path a** (X → M): variant → ARHGAP35_expression, covariates: msex_u, age_death_u, pmi_u
- **Path b** (M → Y): ARHGAP35_expression → tangles_sqrt, covariates: msex_u, age_death_u, educ, apoe4_dose, apoe2_dose
- **Path c'** (X → Y direct): variant → tangles_sqrt, covariates: msex_u, age_death_u, educ, apoe4_dose, apoe2_dose

## Methods
1. **SEM with FIML** (primary) — handles partial overlap / missing data under MAR
2. **Bootstrap** — resamples entire pipeline for robust CI
3. **MNAR Sensitivity** — tipping-point analysis for departures from MAR
4. **Bayesian (blavaan)** — MCMC-based posterior inference

## Data
- N = 1153 total samples
- 113 with expression data, 1067 with tangles_sqrt, 108 with both
- Partial overlap handled by FIML

## Conclusion
 - All four analytical approaches used the full dataset (N=1,153) with missing-data handling (FIML or Bayesian data augmentation) to ensure maximum statistical power.
 - Across all four analytical approaches, the indirect (mediation) effects were consistently small and non-significant for both variants (FIML indirect: 0.044, p=0.205 for the indel; 0.048, p=0.195 for the SNP; all 95% CIs spanning zero). 
 - The direct effects of both variants on tangles were highly significant (c' ≈ −0.33, p<0.001), indicating that the variant–tangle association operates through pathways other than ARHGAP35 expression in Ast.10 (consistent with bootstrap with FIML and Bayesian blavaan sensitivity analyses on the full dataset).
 - The MNAR sensitivity analysis showed that even moderate departures from the missing-at-random assumption would not change this conclusion.
 - Together, these results provide evidence that ARHGAP35 expression in Ast.10 does not mediate the set 57 genetic effects on tangle burden.

## 0. Setup and Data Loading

In [ ]:
# ── Libraries ──────────────────────────────────────────────────────────────
suppressPackageStartupMessages({
  library(lavaan)
  library(ggplot2)
  library(dplyr)
  library(tidyr)
  library(gridExtra)
  library(boot)
})

# ── Paths ──────────────────────────────────────────────────────────────────
base_dir   <- "/mnt/lustre/home/yl4437/xqtl_flagship/APOE/mediation_partialoverlap_claude"
input_file <- file.path(base_dir, "APOE_ind_set_57_mediation_all_input.txt")
fiml_dir   <- file.path(base_dir, "main_SEM_FIML")
boot_dir   <- file.path(base_dir, "bootstrap")
mnar_dir   <- file.path(base_dir, "MNAR_sensitivity")
bayes_dir  <- file.path(base_dir, "bayesian_blavaan")
summ_dir   <- file.path(base_dir, "summary")

# ── Read data ─────────────────────────────────────────────────────────────
dat <- read.delim(input_file, stringsAsFactors = FALSE)
cat("Total samples:", nrow(dat), "\n")
cat("Non-NA expression:", sum(!is.na(dat$ARHGAP35_expression)), "\n")
cat("Non-NA tangles_sqrt:", sum(!is.na(dat$tangles_sqrt)), "\n")
cat("Non-NA both (expression & tangles):", sum(!is.na(dat$ARHGAP35_expression) & !is.na(dat$tangles_sqrt)), "\n")

# ── Define exposure variants ──────────────────────────────────────────────
exposures <- c("chr19_44827033_TGATAGATAGATAGATA_TGATA", "chr19_44840322_G_A")
mediator  <- "ARHGAP35_expression"
outcome   <- "tangles_sqrt"

# Covariates for each path
cov_xm <- c("msex_u", "age_death_u", "pmi_u")                          # X → M
cov_my <- c("msex_u", "age_death_u", "educ", "apoe4_dose", "apoe2_dose") # M → Y and X → Y

cat("\nExposures:", exposures, "\n")
cat("Mediator:", mediator, "\n")
cat("Outcome:", outcome, "\n")

## 1. Primary Analysis: SEM with FIML

In [ ]:
# ── Function to build lavaan model syntax for one exposure ─────────────────
build_mediation_syntax <- function(exposure) {
  # Path a: M ~ X + covariates_xm
  path_a <- paste0(mediator, " ~ a*", exposure, " + ",
                   paste(cov_xm, collapse = " + "))
  # Path b + c': Y ~ M + X + covariates_my
  path_bc <- paste0(outcome, " ~ b*", mediator, " + c_prime*", exposure, " + ",
                    paste(cov_my, collapse = " + "))
  # Defined parameters
  defs <- c(
    "indirect := a*b",
    "total    := c_prime + a*b",
    "prop_med := (a*b) / (c_prime + a*b)"
  )
  paste(c(path_a, path_bc, defs), collapse = "\n")
}

cat("Model syntax for exposure 1:\n")
cat(build_mediation_syntax(exposures[1]), "\n")

In [ ]:
# ── Run FIML for both exposures ─────────────────────────────────────────────
fiml_results_list <- list()

for (exp_var in exposures) {
  cat("\n========================================================\n")
  cat("FIML: Exposure =", exp_var, "\n")
  cat("========================================================\n")
  
  mod_syntax <- build_mediation_syntax(exp_var)
  
  fit <- sem(mod_syntax, data = dat, missing = "fiml.x",
             fixed.x = FALSE, estimator = "ML")
  
  cat("\nModel fit summary:\n")
  print(summary(fit, fit.measures = TRUE, ci = TRUE, standardized = TRUE))
  
  # Extract all parameters
  params <- parameterEstimates(fit, ci = TRUE, standardized = TRUE)
  
  # Fit measures
  fm <- fitMeasures(fit, c("chisq", "df", "pvalue", "cfi", "rmsea", "srmr",
                           "aic", "bic", "logl", "npar"))
  
  # Key paths
  key_labels <- c("a", "b", "c_prime", "indirect", "total", "prop_med")
  key <- params[params$label %in% key_labels,
                c("label", "est", "se", "z", "pvalue", "ci.lower", "ci.upper")]
  key$CI_95 <- sprintf("[%.4f, %.4f]", key$ci.lower, key$ci.upper)
  key$sig <- ifelse(key$pvalue < 0.001, "***",
             ifelse(key$pvalue < 0.01, "**",
             ifelse(key$pvalue < 0.05, "*",
             ifelse(key$pvalue < 0.1, ".", ""))))
  
  cat("\nKey path estimates:\n")
  print(key)
  
  fiml_results_list[[exp_var]] <- list(
    fit = fit, params = params, fit_measures = fm, key = key, exposure = exp_var
  )
}

cat("\nFIML analysis complete for both exposures.\n")

In [ ]:
# ── Save FIML results ───────────────────────────────────────────────────────
for (exp_var in exposures) {
  res <- fiml_results_list[[exp_var]]
  prefix <- gsub("chr19_", "", exp_var)
  
  # All parameters
  write.csv(res$params, file.path(fiml_dir, paste0("fiml_all_params_", prefix, ".csv")),
            row.names = FALSE)
  # Key paths
  write.csv(res$key, file.path(fiml_dir, paste0("fiml_key_paths_", prefix, ".csv")),
            row.names = FALSE)
  # Fit measures — convert named vector to single-row data.frame safely
  fm_df <- data.frame(as.list(res$fit_measures))
  write.csv(fm_df, file.path(fiml_dir, paste0("fiml_fit_measures_", prefix, ".csv")),
            row.names = FALSE)
  
  # Summary table
  summ_tbl <- res$key[, c("label", "est", "se", "pvalue", "CI_95", "sig")]
  write.csv(summ_tbl, file.path(fiml_dir, paste0("fiml_summary_table_", prefix, ".csv")),
            row.names = FALSE)
}

cat("FIML results saved to:", fiml_dir, "\n")

In [ ]:
# ── FIML Forest Plot ────────────────────────────────────────────────────────
fiml_plot_data <- do.call(rbind, lapply(exposures, function(exp_var) {
  res <- fiml_results_list[[exp_var]]
  key <- res$key
  key$exposure <- exp_var
  key
}))

fiml_plot_data$label <- factor(fiml_plot_data$label,
                               levels = c("a", "b", "c_prime", "indirect", "total", "prop_med"))
fiml_plot_data$exposure_short <- ifelse(
  grepl("44827033", fiml_plot_data$exposure), "44827033_indel", "44840322_G_A")

p_fiml <- ggplot(fiml_plot_data, aes(x = est, y = label, color = exposure_short)) +
  geom_vline(xintercept = 0, linetype = "dashed", color = "grey50") +
  geom_pointrange(aes(xmin = ci.lower, xmax = ci.upper),
                  position = position_dodge(width = 0.5), size = 0.6) +
  labs(title = "FIML SEM: Path Estimates",
       subtitle = "ARHGAP35 expression mediating variant → tangles_sqrt",
       x = "Estimate (95% CI)", y = "Path", color = "Exposure") +
  theme_minimal(base_size = 12) +
  theme(legend.position = "bottom")

ggsave(file.path(fiml_dir, "fiml_forest_plot.png"), p_fiml, width = 8, height = 5, dpi = 150)
ggsave(file.path(fiml_dir, "fiml_forest_plot.pdf"), p_fiml, width = 8, height = 5)
print(p_fiml)
cat("FIML forest plot saved.\n")

## 2. Sensitivity Analysis: Bootstrap

In [ ]:
# ── Bootstrap mediation (FIML + bootstrap) ───────────────────────────────────
# lavaan supports bootstrap with missing="fiml.x", so we use the full dataset
# (N=1153) rather than restricting to complete cases (N=107).

set.seed(12345)
n_boot <- 2000

boot_results_list <- list()

for (exp_var in exposures) {
  cat("\nBootstrap: Exposure =", exp_var, "\n")
  
  mod_syntax <- build_mediation_syntax(exp_var)
  
  fit_boot <- sem(mod_syntax, data = dat, missing = "fiml.x",
                  fixed.x = FALSE, se = "bootstrap",
                  bootstrap = n_boot, iseed = 12345)
  
  # BCa bootstrap CIs
  params_boot <- parameterEstimates(fit_boot, boot.ci.type = "bca.simple",
                                     level = 0.95, ci = TRUE)
  
  key_labels <- c("a", "b", "c_prime", "indirect", "total", "prop_med")
  key_boot <- params_boot[params_boot$label %in% key_labels,
                          c("label", "est", "se", "z", "pvalue", "ci.lower", "ci.upper")]
  key_boot$CI_95 <- sprintf("[%.4f, %.4f]", key_boot$ci.lower, key_boot$ci.upper)
  key_boot$sig <- ifelse(key_boot$pvalue < 0.001, "***",
                  ifelse(key_boot$pvalue < 0.01, "**",
                  ifelse(key_boot$pvalue < 0.05, "*",
                  ifelse(key_boot$pvalue < 0.1, ".", ""))))
  
  cat("Key bootstrap results:\n")
  print(key_boot)
  
  # Extract bootstrap distribution of indirect effect
  boot_samples <- lavInspect(fit_boot, what = "boot")
  
  boot_results_list[[exp_var]] <- list(
    fit = fit_boot, key = key_boot, params = params_boot,
    boot_samples = boot_samples, exposure = exp_var
  )
}

cat("\nBootstrap analysis complete (N =", nrow(dat), "with FIML).\n")

In [ ]:
# ── Save bootstrap results ───────────────────────────────────────────────────
for (exp_var in exposures) {
  res <- boot_results_list[[exp_var]]
  prefix <- gsub("chr19_", "", exp_var)
  
  write.csv(res$key, file.path(boot_dir, paste0("bootstrap_results_", prefix, ".csv")),
            row.names = FALSE)
  write.csv(as.data.frame(res$boot_samples),
            file.path(boot_dir, paste0("bootstrap_distributions_", prefix, ".csv")),
            row.names = FALSE)
}

# ── Bootstrap distribution plots ─────────────────────────────────────────────
for (exp_var in exposures) {
  res <- boot_results_list[[exp_var]]
  prefix <- gsub("chr19_", "", exp_var)
  
  # Get the indirect effect column from bootstrap samples
  boot_mat <- as.data.frame(res$boot_samples)
  # The defined parameters are the last columns
  params_est <- parameterEstimates(res$fit)
  # Find the column index for indirect effect
  free_params <- params_est[params_est$op != ":=", ]
  def_params <- params_est[params_est$op == ":=", ]
  
  # Bootstrap samples: free params first, then defined params appended
  n_free <- nrow(free_params)
  if (ncol(boot_mat) > n_free) {
    indirect_col <- n_free + which(def_params$label == "indirect")
    indirect_boot <- boot_mat[, indirect_col]
  } else {
    # Compute from a and b columns
    a_idx <- which(free_params$label == "a")
    b_idx <- which(free_params$label == "b")
    indirect_boot <- boot_mat[, a_idx] * boot_mat[, b_idx]
  }
  
  p_boot <- ggplot(data.frame(indirect = indirect_boot), aes(x = indirect)) +
    geom_histogram(bins = 50, fill = "steelblue", alpha = 0.7, color = "white") +
    geom_vline(xintercept = 0, linetype = "dashed", color = "red", linewidth = 0.8) +
    geom_vline(xintercept = median(indirect_boot), linetype = "solid", color = "darkblue") +
    labs(title = paste("Bootstrap Distribution of Indirect Effect"),
         subtitle = paste("Exposure:", prefix),
         x = "Indirect Effect (a x b)", y = "Count") +
    theme_minimal(base_size = 12)
  
  ggsave(file.path(boot_dir, paste0("bootstrap_distributions_", prefix, ".png")),
         p_boot, width = 7, height = 5, dpi = 150)
  ggsave(file.path(boot_dir, paste0("bootstrap_distributions_", prefix, ".pdf")),
         p_boot, width = 7, height = 5)
  print(p_boot)
}

# ── Bootstrap forest plot (both exposures) ─────────────────────────────────
boot_plot_data <- do.call(rbind, lapply(exposures, function(exp_var) {
  res <- boot_results_list[[exp_var]]
  key <- res$key
  key$exposure <- exp_var
  key
}))
boot_plot_data$label <- factor(boot_plot_data$label,
                               levels = c("a", "b", "c_prime", "indirect", "total", "prop_med"))
boot_plot_data$exposure_short <- ifelse(
  grepl("44827033", boot_plot_data$exposure), "44827033_indel", "44840322_G_A")

p_boot_forest <- ggplot(boot_plot_data, aes(x = est, y = label, color = exposure_short)) +
  geom_vline(xintercept = 0, linetype = "dashed", color = "grey50") +
  geom_pointrange(aes(xmin = ci.lower, xmax = ci.upper),
                  position = position_dodge(width = 0.5), size = 0.6) +
  labs(title = "Bootstrap (BCa + FIML): Path Estimates",
       subtitle = paste0("N = ", nrow(dat), " (FIML), ", n_boot, " resamples"),
       x = "Estimate (95% BCa CI)", y = "Path", color = "Exposure") +
  theme_minimal(base_size = 12) +
  theme(legend.position = "bottom")

ggsave(file.path(boot_dir, "bootstrap_forest_plot.png"), p_boot_forest, width = 8, height = 5, dpi = 150)
ggsave(file.path(boot_dir, "bootstrap_forest_plot.pdf"), p_boot_forest, width = 8, height = 5)
print(p_boot_forest)
cat("Bootstrap results saved to:", boot_dir, "\n")

## 3. Sensitivity Analysis: MNAR Tipping-Point

In [ ]:
# ── MNAR Sensitivity Analysis ────────────────────────────────────────────────
# Under MNAR, missing expression/outcome values could be systematically shifted.
# We add offsets (delta_M, delta_Y) to the observed mediator/outcome for subjects
# who would be missing (simulating what those values "would have been").
# Then re-run FIML and see if conclusions change.
#
# Here we do a grid: shift the *observed* values to simulate what the analysis
# would yield under various MNAR departures.

delta_m_grid <- seq(-1.0, 1.0, by = 0.25)
delta_y_grid <- seq(-1.0, 1.0, by = 0.25)

mnar_results_all <- list()

for (exp_var in exposures) {
  cat("\nMNAR sensitivity: Exposure =", exp_var, "\n")
  mod_syntax <- build_mediation_syntax(exp_var)
  
  grid_results <- expand.grid(delta_m = delta_m_grid, delta_y = delta_y_grid,
                              a = NA_real_, b = NA_real_, indirect = NA_real_,
                              indirect_p = NA_real_, c_prime = NA_real_,
                              total = NA_real_, total_p = NA_real_,
                              stringsAsFactors = FALSE)
  
  for (i in seq_len(nrow(grid_results))) {
    dm <- grid_results$delta_m[i]
    dy <- grid_results$delta_y[i]
    
    dat_shifted <- dat
    dat_shifted[[mediator]] <- dat_shifted[[mediator]] + dm
    dat_shifted[[outcome]]  <- dat_shifted[[outcome]]  + dy
    
    fit_s <- tryCatch(
      sem(mod_syntax, data = dat_shifted, missing = "fiml.x",
          fixed.x = FALSE, estimator = "ML"),
      error = function(e) NULL
    )
    
    if (!is.null(fit_s) && lavInspect(fit_s, "converged")) {
      pe <- parameterEstimates(fit_s)
      get_val <- function(lbl, col) {
        row <- pe[pe$label == lbl, ]
        if (nrow(row) > 0) row[[col]][1] else NA
      }
      grid_results$a[i]          <- get_val("a", "est")
      grid_results$b[i]          <- get_val("b", "est")
      grid_results$indirect[i]   <- get_val("indirect", "est")
      grid_results$indirect_p[i] <- get_val("indirect", "pvalue")
      grid_results$c_prime[i]    <- get_val("c_prime", "est")
      grid_results$total[i]      <- get_val("total", "est")
      grid_results$total_p[i]    <- get_val("total", "pvalue")
    }
  }
  
  grid_results$exposure <- exp_var
  mnar_results_all[[exp_var]] <- grid_results
  
  cat("  Grid complete:", sum(!is.na(grid_results$indirect)), "/",
      nrow(grid_results), "converged\n")
}

mnar_all <- do.call(rbind, mnar_results_all)
cat("\nMNAR sensitivity analysis complete.\n")

In [ ]:
# ── Save MNAR results ────────────────────────────────────────────────────────
write.csv(mnar_all, file.path(mnar_dir, "mnar_sensitivity_grid.csv"), row.names = FALSE)

# ── Find tipping points (where indirect effect crosses 0 or becomes non-significant)
# Reference: FIML indirect at delta_m=0, delta_y=0
tipping_points <- list()
for (exp_var in exposures) {
  gr <- mnar_results_all[[exp_var]]
  ref_indirect <- gr$indirect[gr$delta_m == 0 & gr$delta_y == 0]
  ref_sign <- sign(ref_indirect)
  
  # Points where sign flips
  sign_flips <- gr[sign(gr$indirect) != ref_sign & !is.na(gr$indirect), ]
  # Points where p > 0.05 (becomes non-significant)
  nonsig <- gr[gr$indirect_p > 0.05 & !is.na(gr$indirect_p), ]
  
  if (nrow(sign_flips) > 0) {
    closest_flip <- sign_flips[which.min(abs(sign_flips$delta_m) + abs(sign_flips$delta_y)), ]
  } else {
    closest_flip <- data.frame(delta_m = NA, delta_y = NA, indirect = NA)
  }
  
  tipping_points[[exp_var]] <- data.frame(
    exposure = exp_var,
    ref_indirect = ref_indirect,
    sign_flip_delta_m = closest_flip$delta_m[1],
    sign_flip_delta_y = closest_flip$delta_y[1],
    n_nonsig_of_total = paste0(nrow(nonsig), "/", nrow(gr))
  )
}

tipping_df <- do.call(rbind, tipping_points)
write.csv(tipping_df, file.path(mnar_dir, "mnar_tipping_points.csv"), row.names = FALSE)
cat("Tipping points:\n")
print(tipping_df)

In [ ]:
# ── MNAR contour plots ───────────────────────────────────────────────────────
for (exp_var in exposures) {
  gr <- mnar_results_all[[exp_var]]
  prefix <- gsub("chr19_", "", exp_var)
  
  # Contour: indirect effect
  p_contour_ind <- ggplot(gr, aes(x = delta_m, y = delta_y, z = indirect)) +
    geom_contour_filled(bins = 12) +
    geom_point(data = gr[gr$delta_m == 0 & gr$delta_y == 0, ],
               aes(x = delta_m, y = delta_y), color = "red", size = 3, shape = 4, stroke = 2) +
    labs(title = paste("MNAR Sensitivity: Indirect Effect"),
         subtitle = paste("Exposure:", prefix),
         x = expression(delta[M]~"(mediator shift)"),
         y = expression(delta[Y]~"(outcome shift)"),
         fill = "Indirect\nEffect") +
    theme_minimal(base_size = 12)
  
  ggsave(file.path(mnar_dir, paste0("mnar_contour_indirect_", prefix, ".png")),
         p_contour_ind, width = 7, height = 6, dpi = 150)
  ggsave(file.path(mnar_dir, paste0("mnar_contour_indirect_", prefix, ".pdf")),
         p_contour_ind, width = 7, height = 6)
  print(p_contour_ind)
  
  # Contour: p-value of indirect effect
  gr$log10p <- -log10(pmax(gr$indirect_p, 1e-20))
  p_contour_p <- ggplot(gr, aes(x = delta_m, y = delta_y, z = log10p)) +
    geom_contour_filled(bins = 12) +
    geom_point(data = gr[gr$delta_m == 0 & gr$delta_y == 0, ],
               aes(x = delta_m, y = delta_y), color = "red", size = 3, shape = 4, stroke = 2) +
    labs(title = paste("MNAR Sensitivity: -log10(p) of Indirect Effect"),
         subtitle = paste("Exposure:", prefix),
         x = expression(delta[M]~"(mediator shift)"),
         y = expression(delta[Y]~"(outcome shift)"),
         fill = "-log10(p)") +
    theme_minimal(base_size = 12)
  
  ggsave(file.path(mnar_dir, paste0("mnar_contour_pvalue_", prefix, ".png")),
         p_contour_p, width = 7, height = 6, dpi = 150)
  ggsave(file.path(mnar_dir, paste0("mnar_contour_pvalue_", prefix, ".pdf")),
         p_contour_p, width = 7, height = 6)
  print(p_contour_p)
}

# ── 1D slices at delta_y=0 and delta_m=0 ─────────────────────────────────
slice_plots <- list()
for (exp_var in exposures) {
  gr <- mnar_results_all[[exp_var]]
  prefix <- gsub("chr19_", "", exp_var)
  
  slice_m <- gr[gr$delta_y == 0, ]
  slice_y <- gr[gr$delta_m == 0, ]
  
  p1 <- ggplot(slice_m, aes(x = delta_m, y = indirect)) +
    geom_line(color = "steelblue", linewidth = 1) +
    geom_point(color = "steelblue") +
    geom_hline(yintercept = 0, linetype = "dashed", color = "red") +
    labs(title = paste0(prefix, ": Indirect vs delta_M"),
         x = expression(delta[M]), y = "Indirect Effect") +
    theme_minimal(base_size = 11)
  
  p2 <- ggplot(slice_y, aes(x = delta_y, y = indirect)) +
    geom_line(color = "darkorange", linewidth = 1) +
    geom_point(color = "darkorange") +
    geom_hline(yintercept = 0, linetype = "dashed", color = "red") +
    labs(title = paste0(prefix, ": Indirect vs delta_Y"),
         x = expression(delta[Y]), y = "Indirect Effect") +
    theme_minimal(base_size = 11)
  
  p_combined <- grid.arrange(p1, p2, ncol = 2)
  
  ggsave(file.path(mnar_dir, paste0("mnar_1d_slices_", prefix, ".png")),
         p_combined, width = 10, height = 4, dpi = 150)
  ggsave(file.path(mnar_dir, paste0("mnar_1d_slices_", prefix, ".pdf")),
         p_combined, width = 10, height = 4)
}

cat("MNAR sensitivity results saved to:", mnar_dir, "\n")

## 4. Sensitivity Analysis: Bayesian (blavaan)

In [ ]:
# ── Bayesian mediation via blavaan ───────────────────────────────────────────
# blavaan handles missing data natively (similar to FIML), so we use the full
# dataset (N=1153) rather than restricting to complete cases.
suppressPackageStartupMessages(library(blavaan))

bayes_results_list <- list()

for (exp_var in exposures) {
  cat("\n========================================================\n")
  cat("Bayesian: Exposure =", exp_var, "\n")
  cat("========================================================\n")
  
  mod_syntax <- build_mediation_syntax(exp_var)
  
  fit_bayes <- bsem(mod_syntax, data = dat,
                    n.chains = 4, burnin = 2000, sample = 5000,
                    seed = 12345)
  
  cat("\nBayesian model summary:\n")
  print(summary(fit_bayes))
  
  # Extract MCMC draws
  mcmc_draws <- blavInspect(fit_bayes, what = "mcmc")
  mcmc_mat <- do.call(rbind, mcmc_draws)
  
  # blavaan parameterEstimates has limited columns; compute summaries from MCMC
  pe <- parameterEstimates(fit_bayes)
  
  # Map key labels to MCMC column names
  key_labels <- c("a", "b", "c_prime", "indirect", "total", "prop_med")
  mcmc_cols <- colnames(mcmc_mat)
  
  key_rows <- list()
  for (lbl in key_labels) {
    if (lbl %in% mcmc_cols) {
      samps <- mcmc_mat[, lbl]
    } else if (lbl == "indirect" && "a" %in% mcmc_cols && "b" %in% mcmc_cols) {
      samps <- mcmc_mat[, "a"] * mcmc_mat[, "b"]
    } else if (lbl == "total" && "a" %in% mcmc_cols && "b" %in% mcmc_cols && "c_prime" %in% mcmc_cols) {
      samps <- mcmc_mat[, "c_prime"] + mcmc_mat[, "a"] * mcmc_mat[, "b"]
    } else if (lbl == "prop_med" && "a" %in% mcmc_cols && "b" %in% mcmc_cols && "c_prime" %in% mcmc_cols) {
      ab <- mcmc_mat[, "a"] * mcmc_mat[, "b"]
      tot <- mcmc_mat[, "c_prime"] + ab
      samps <- ifelse(abs(tot) > 1e-10, ab / tot, NA)
      samps <- samps[!is.na(samps)]
    } else {
      next
    }
    key_rows[[lbl]] <- data.frame(
      label = lbl,
      est = mean(samps),
      se = sd(samps),
      ci.lower = quantile(samps, 0.025, names = FALSE),
      ci.upper = quantile(samps, 0.975, names = FALSE),
      stringsAsFactors = FALSE
    )
  }
  key_bayes <- do.call(rbind, key_rows)
  key_bayes$CI_95 <- sprintf("[%.4f, %.4f]", key_bayes$ci.lower, key_bayes$ci.upper)
  
  cat("\nKey Bayesian results:\n")
  print(key_bayes)
  
  bayes_results_list[[exp_var]] <- list(
    fit = fit_bayes, key = key_bayes, params = pe,
    mcmc_draws = mcmc_draws, mcmc_mat = mcmc_mat, exposure = exp_var
  )
}

cat("\nBayesian analysis complete (N =", nrow(dat), "with missing data handling).\n")

In [ ]:
# ── Save Bayesian results ────────────────────────────────────────────────────
for (exp_var in exposures) {
  res <- bayes_results_list[[exp_var]]
  prefix <- gsub("chr19_", "", exp_var)
  mcmc_mat <- res$mcmc_mat
  
  write.csv(res$key, file.path(bayes_dir, paste0("bayesian_results_", prefix, ".csv")),
            row.names = FALSE)
  write.csv(res$params, file.path(bayes_dir, paste0("bayesian_all_params_", prefix, ".csv")),
            row.names = FALSE)
  
  # Diagnostics: mean, sd, quantiles per MCMC column
  diag_df <- data.frame(
    parameter = colnames(mcmc_mat),
    mean = colMeans(mcmc_mat),
    sd = apply(mcmc_mat, 2, sd),
    q2.5 = apply(mcmc_mat, 2, quantile, 0.025),
    q97.5 = apply(mcmc_mat, 2, quantile, 0.975),
    row.names = NULL
  )
  write.csv(diag_df, file.path(bayes_dir, paste0("bayesian_diagnostics_", prefix, ".csv")),
            row.names = FALSE)
  
  # Save MCMC draws for derived parameters
  mcmc_derived <- data.frame(
    a = mcmc_mat[, "a"],
    b = mcmc_mat[, "b"],
    c_prime = mcmc_mat[, "c_prime"],
    indirect = mcmc_mat[, "a"] * mcmc_mat[, "b"],
    total = mcmc_mat[, "c_prime"] + mcmc_mat[, "a"] * mcmc_mat[, "b"]
  )
  write.csv(mcmc_derived, file.path(bayes_dir, paste0("bayesian_mcmc_derived_", prefix, ".csv")),
            row.names = FALSE)
}

cat("Bayesian results saved to:", bayes_dir, "\n")

In [ ]:
# ── Bayesian posterior plots ──────────────────────────────────────────────────
for (exp_var in exposures) {
  res <- bayes_results_list[[exp_var]]
  prefix <- gsub("chr19_", "", exp_var)
  mcmc_mat <- res$mcmc_mat
  
  a_post <- mcmc_mat[, "a"]
  b_post <- mcmc_mat[, "b"]
  c_post <- mcmc_mat[, "c_prime"]
  indirect_post <- a_post * b_post
  
  post_df <- data.frame(
    value = c(a_post, b_post, c_post, indirect_post),
    path = rep(c("a (X->M)", "b (M->Y)", "c' (X->Y direct)", "indirect (a*b)"),
               each = length(a_post))
  )
  
  p_post <- ggplot(post_df, aes(x = value, fill = path)) +
    geom_density(alpha = 0.5) +
    geom_vline(xintercept = 0, linetype = "dashed", color = "red") +
    facet_wrap(~path, scales = "free") +
    labs(title = paste("Bayesian Posterior Distributions"),
         subtitle = paste("Exposure:", prefix),
         x = "Parameter Value", y = "Density") +
    theme_minimal(base_size = 12) +
    theme(legend.position = "none")
  
  ggsave(file.path(bayes_dir, paste0("bayesian_posteriors_", prefix, ".png")),
         p_post, width = 10, height = 7, dpi = 150)
  ggsave(file.path(bayes_dir, paste0("bayesian_posteriors_", prefix, ".pdf")),
         p_post, width = 10, height = 7)
  print(p_post)
  
  # Trace plot for path a
  n_per_chain <- length(a_post) / 4
  trace_df <- data.frame(
    iteration = seq_along(a_post),
    a = a_post,
    chain = rep(1:4, each = n_per_chain)
  )
  p_trace <- ggplot(trace_df, aes(x = iteration, y = a, color = factor(chain))) +
    geom_line(alpha = 0.5, linewidth = 0.3) +
    labs(title = paste("MCMC Trace: Path a (X -> M)"),
         subtitle = paste("Exposure:", prefix),
         x = "Iteration", y = "a", color = "Chain") +
    theme_minimal(base_size = 12)
  
  ggsave(file.path(bayes_dir, paste0("bayesian_trace_a_", prefix, ".png")),
         p_trace, width = 8, height = 4, dpi = 150)
  ggsave(file.path(bayes_dir, paste0("bayesian_trace_a_", prefix, ".pdf")),
         p_trace, width = 8, height = 4)
  print(p_trace)
}

# ── Bayesian forest plot ──────────────────────────────────────────────────
bayes_plot_data <- do.call(rbind, lapply(exposures, function(exp_var) {
  res <- bayes_results_list[[exp_var]]
  key <- res$key
  key$exposure <- exp_var
  key
}))
bayes_plot_data$label <- factor(bayes_plot_data$label,
                                levels = c("a", "b", "c_prime", "indirect", "total", "prop_med"))
bayes_plot_data$exposure_short <- ifelse(
  grepl("44827033", bayes_plot_data$exposure), "44827033_indel", "44840322_G_A")

p_bayes_forest <- ggplot(bayes_plot_data, aes(x = est, y = label, color = exposure_short)) +
  geom_vline(xintercept = 0, linetype = "dashed", color = "grey50") +
  geom_pointrange(aes(xmin = ci.lower, xmax = ci.upper),
                  position = position_dodge(width = 0.5), size = 0.6) +
  labs(title = "Bayesian (blavaan): Path Estimates",
       subtitle = paste0("N = ", nrow(dat), " (missing data handled), 4 chains x 5000 samples"),
       x = "Posterior Mean (95% Credible Interval)", y = "Path", color = "Exposure") +
  theme_minimal(base_size = 12) +
  theme(legend.position = "bottom")

ggsave(file.path(bayes_dir, "bayesian_forest_plot.png"), p_bayes_forest, width = 8, height = 5, dpi = 150)
ggsave(file.path(bayes_dir, "bayesian_forest_plot.pdf"), p_bayes_forest, width = 8, height = 5)
print(p_bayes_forest)
cat("Bayesian plots saved to:", bayes_dir, "\n")

## 5. Combined Summary: All Methods

In [ ]:
# ── Combine all results into one summary table ────────────────────────────────
combine_key <- function(method_name, results_list, ci_type, n_note) {
  do.call(rbind, lapply(exposures, function(exp_var) {
    res <- results_list[[exp_var]]
    key <- res$key
    data.frame(
      exposure = exp_var,
      method = method_name,
      label = key$label,
      est = key$est,
      se = key$se,
      ci_lower = key$ci.lower,
      ci_upper = key$ci.upper,
      p_value = if ("pvalue" %in% names(key)) key$pvalue else NA_real_,
      ci_type = ci_type,
      n_eff = n_note,
      stringsAsFactors = FALSE
    )
  }))
}

n_all <- paste0("N=", nrow(dat), " (FIML/missing data)")

summary_all <- rbind(
  combine_key("FIML", fiml_results_list, "Delta-method 95% CI", n_all),
  combine_key("Bootstrap (BCa)", boot_results_list, "BCa bootstrap 95% CI", n_all),
  combine_key("Bayesian (blavaan)", bayes_results_list, "95% Credible Interval", n_all)
)

write.csv(summary_all, file.path(summ_dir, "all_methods_summary.csv"), row.names = FALSE)
cat("Combined summary saved. Rows:", nrow(summary_all), "\n")
print(summary_all)

In [ ]:
# ── Summary forest plot: all methods, by exposure ─────────────────────────────
summary_all$label <- factor(summary_all$label,
                            levels = c("a", "b", "c_prime", "indirect", "total", "prop_med"))
summary_all$exposure_short <- ifelse(
  grepl("44827033", summary_all$exposure), "44827033_indel", "44840322_G_A")

# ── Plot 1: Forest plot faceted by exposure, showing all methods per path ──
p_forest_all <- ggplot(summary_all, aes(x = est, y = label, color = method, shape = method)) +
  geom_vline(xintercept = 0, linetype = "dashed", color = "grey50") +
  geom_pointrange(aes(xmin = ci_lower, xmax = ci_upper),
                  position = position_dodge(width = 0.6), size = 0.5) +
  facet_wrap(~exposure_short, ncol = 1, scales = "free_x") +
  labs(title = "All Methods Comparison: Path Estimates",
       subtitle = "ARHGAP35 expression mediating variant → tangles_sqrt",
       x = "Estimate (95% CI)", y = "Path",
       color = "Method", shape = "Method") +
  theme_minimal(base_size = 12) +
  theme(legend.position = "bottom")

ggsave(file.path(summ_dir, "summary_forest_all_methods.png"), p_forest_all,
       width = 10, height = 8, dpi = 150)
ggsave(file.path(summ_dir, "summary_forest_all_methods.pdf"), p_forest_all,
       width = 10, height = 8)
print(p_forest_all)

In [ ]:
# ── Plot 2: Faceted by path (each effect type gets its own panel) ─────────
p_facet_path <- ggplot(summary_all, aes(x = est, y = method, color = exposure_short)) +
  geom_vline(xintercept = 0, linetype = "dashed", color = "grey50") +
  geom_pointrange(aes(xmin = ci_lower, xmax = ci_upper),
                  position = position_dodge(width = 0.5), size = 0.5) +
  facet_wrap(~label, scales = "free_x", ncol = 2) +
  labs(title = "Estimates by Effect Type Across Methods",
       subtitle = "ARHGAP35 expression mediating variant → tangles_sqrt",
       x = "Estimate (95% CI)", y = "Method", color = "Exposure") +
  theme_minimal(base_size = 11) +
  theme(legend.position = "bottom",
        strip.text = element_text(face = "bold"))

ggsave(file.path(summ_dir, "summary_faceted_by_path.png"), p_facet_path,
       width = 10, height = 8, dpi = 150)
ggsave(file.path(summ_dir, "summary_faceted_by_path.pdf"), p_facet_path,
       width = 10, height = 8)
print(p_facet_path)

In [ ]:
# ── Plot 3: Indirect effect comparison (focused) ─────────────────────────────
indirect_data <- summary_all[summary_all$label == "indirect", ]

p_indirect <- ggplot(indirect_data, aes(x = est, y = method, color = exposure_short)) +
  geom_vline(xintercept = 0, linetype = "dashed", color = "grey50") +
  geom_pointrange(aes(xmin = ci_lower, xmax = ci_upper),
                  position = position_dodge(width = 0.4), size = 0.8) +
  labs(title = "Indirect Effect (a x b) Across Methods",
       subtitle = "Mediation: variant → ARHGAP35 expression → tangles_sqrt",
       x = "Indirect Effect (95% CI)", y = "Method", color = "Exposure") +
  theme_minimal(base_size = 12) +
  theme(legend.position = "bottom")

ggsave(file.path(summ_dir, "summary_indirect_effect.png"), p_indirect,
       width = 8, height = 4, dpi = 150)
ggsave(file.path(summ_dir, "summary_indirect_effect.pdf"), p_indirect,
       width = 8, height = 4)
print(p_indirect)

In [ ]:
# ── Plot 4: Combined panel ───────────────────────────────────────────────────
# Top: forest by exposure; Bottom: indirect effect focused
p_combined_panel <- grid.arrange(
  p_forest_all + theme(legend.position = "right"),
  p_indirect + theme(legend.position = "right"),
  ncol = 1, heights = c(2, 1)
)

ggsave(file.path(summ_dir, "summary_combined_panel.png"), p_combined_panel,
       width = 11, height = 10, dpi = 150)
ggsave(file.path(summ_dir, "summary_combined_panel.pdf"), p_combined_panel,
       width = 11, height = 10)

cat("All summary plots saved to:", summ_dir, "\n")

In [ ]:
# ── Display table for each exposure ──────────────────────────────────────────
for (exp_var in exposures) {
  prefix <- gsub("chr19_", "", exp_var)
  sub_df <- summary_all[summary_all$exposure == exp_var, ]
  
  display_tbl <- sub_df %>%
    mutate(estimate_ci = sprintf("%.4f [%.4f, %.4f]", est, ci_lower, ci_upper)) %>%
    select(method, label, estimate_ci, p_value, ci_type) %>%
    pivot_wider(names_from = method, values_from = c(estimate_ci, p_value, ci_type))
  
  write.csv(display_tbl, file.path(summ_dir, paste0("summary_display_table_", prefix, ".csv")),
            row.names = FALSE)
  
  cat("\n=== Display Table: Exposure =", prefix, "===\n")
  print(display_tbl)
}

cat("\nDisplay tables saved to:", summ_dir, "\n")